# GaitLU-1M shard inspection

Visualize the silhouette sequences behind the GaitLU-1M tar shards.

There are **two** kinds of tar in this pipeline, and they need different code:

| | Raw upstream | Prepared |
|---|---|---|
| Path | `$GAITLU_RAW_ROOT/gaitlu-NNN.tar.gz` | `$GAITLU_PREPARED_ROOT/shards/gaitlu-NNN.tar` |
| Contents | OpenGait pickles at `label/type/view/view.pkl` | `records/<sha256>.bits`, bit-packed |
| Access | stream-only, must unpickle | **seekable** — byte offsets live in `inventories/gaitlu-NNN.csv` |

Prefer the prepared shards. `pack_gaitlu_shard` already recorded byte offsets, so reading a
sequence never touches `tarfile` — just `os.pread` + `np.unpackbits`, mirroring
`GaitLUIndexedDataset._decode` in `src/cody_jepa/data/gaitlu.py`.

Sections:
1. Setup and helpers
2. Prepared shards — contact sheets, animation, grids, QC distributions
3. Raw `.tar.gz` — only for inspecting sequences the packer rejected
4. What the model actually sees — post-transform clips via `GaitLUIndexedDataset`

**Paths.** Set `GAITLU_PREPARED_ROOT` (and `GAITLU_RAW_ROOT` for section 3) to the values used by
`slurm/prepare-gaitlu-shards.sbatch`. On a machine with no prepared root, section 1 tells you so
rather than failing deep inside a plotting call.

## 1. Setup and helpers

In [ ]:
%matplotlib inline

import csv
import io
import os
import pickle
import tarfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

PREPARED = Path(os.environ.get("GAITLU_PREPARED_ROOT", "data/gaitlu-1m/prepared")).expanduser()
RAW = Path(os.environ.get("GAITLU_RAW_ROOT", "data/gaitlu-1m/raw")).expanduser()
SHARD = os.environ.get("GAITLU_SHARD", "gaitlu-000")  # batch runs override this per shard

print("prepared root:", PREPARED, "->", "found" if (PREPARED / "shards").is_dir() else "MISSING")
print("shard:        ", SHARD)
print("raw root:     ", RAW, "->", "found" if RAW.is_dir() else "MISSING")
if (PREPARED / "inventories").is_dir():
    inventories = sorted((PREPARED / "inventories").glob("gaitlu-*.csv"))
    print(f"{len(inventories)} shard inventories, e.g. {[p.stem for p in inventories[:3]]}")

In [ ]:
def load_inventory(shard="gaitlu-000", valid_only=True):
    """Rows from inventories/<shard>.csv, which carries QC columns and exclusion reasons.

    A manifest under manifests/ (common-holdout.csv, ladder-0-small.csv, ...) works with
    read_sequence too: same sequence_id/shard_path/record_offset/record_size/num_frames
    columns, already filtered to eligible sequences.
    """
    path = PREPARED / "inventories" / f"{shard}.csv"
    with path.open(newline="") as handle:
        rows = list(csv.DictReader(handle))
    return [row for row in rows if row["valid"] == "true"] if valid_only else rows


def load_manifest(name="common-holdout.csv"):
    with (PREPARED / "manifests" / name).open(newline="") as handle:
        return list(csv.DictReader(handle))


def find_manifest(split, preferred=None):
    """First manifest whose rows carry the requested split.

    GaitLUIndexedDataset rejects a manifest whose `split` column disagrees with its own
    `split` argument, so a train-side view cannot reuse common-holdout.csv.
    """
    candidates = sorted((PREPARED / "manifests").glob("*.csv"))
    if preferred:
        candidates.sort(key=lambda path: path.name != preferred)
    for path in candidates:
        with path.open(newline="") as handle:
            first = next(csv.DictReader(handle), None)
        if first and first["split"] == split:
            return path
    raise FileNotFoundError(f"no {split!r} manifest under {PREPARED / 'manifests'}")


def read_sequence(row):
    """Random-access one [T, H, W] uint8 silhouette stack. No tar scan, no unpickling."""
    frames = int(row["num_frames"])
    height = int(row["height"])
    width = int(row["width"])
    with (PREPARED / row["shard_path"]).open("rb", buffering=0) as handle:
        payload = os.pread(handle.fileno(), int(row["record_size"]), int(row["record_offset"]))
    if len(payload) != int(row["record_size"]):
        raise OSError(f"short read for {row['sequence_id']!r}")
    bits = np.unpackbits(
        np.frombuffer(payload, dtype=np.uint8), bitorder="little", count=frames * height * width
    )
    return bits.reshape(frames, height, width)

In [ ]:
def show_frame(ax, frame, title=None):
    """Binary masks: pin the range to [0, 1] and disable smoothing, or edges smear."""
    ax.imshow(frame, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    if title:
        ax.set_title(title, fontsize=7)
    ax.axis("off")


def contact_sheet(video, n=8, title=""):
    """Evenly spaced frames from one sequence — the fastest way to eyeball gait."""
    indices = np.linspace(0, len(video) - 1, min(n, len(video))).astype(int)
    fig, axes = plt.subplots(1, len(indices), figsize=(1.4 * len(indices), 2.4))
    for ax, index in zip(np.atleast_1d(axes), indices):
        show_frame(ax, video[index], f"t={index}")
    fig.suptitle(title, fontsize=9)
    fig.tight_layout()
    return fig


def animate(video, fps=10):
    """Inline scrubber. Needs the notebook frontend; falls back to a contact sheet."""
    from matplotlib import animation
    from IPython.display import HTML

    fig, ax = plt.subplots(figsize=(2, 3))
    ax.axis("off")
    image = ax.imshow(video[0], cmap="gray", vmin=0, vmax=1, interpolation="nearest")

    def update(index):
        image.set_data(video[index])
        return (image,)  # blit=True requires the artists back, not set_data's None

    anim = animation.FuncAnimation(
        fig, update, frames=len(video), interval=1000 / fps, blit=True
    )
    plt.close(fig)
    return HTML(anim.to_jshtml())

## 2. Prepared shards

Start with one shard's inventory. Every row here is a sequence the packer accepted, with its
byte offset into `shards/gaitlu-NNN.tar`.

In [ ]:
rows = load_inventory(SHARD)
print(f"{len(rows)} valid sequences in {SHARD}")
for row in rows[:5]:
    print(
        f"  {row['sequence_id']:<40} {row['num_frames']:>4} frames  "
        f"{row['height']}x{row['width']}  fg={float(row['foreground_fraction']):.4f}"
    )

### One sequence: contact sheet

In [ ]:
row = rows[0]
video = read_sequence(row)
print(row["sequence_id"], video.shape, video.dtype, "unique values:", np.unique(video))
contact_sheet(
    video,
    title=f"{row['sequence_id']}  fg={float(row['foreground_fraction']):.3f}  "
    f"empty={float(row['empty_frame_fraction']):.3f}",
);

### One sequence: animation

In [ ]:
animate(read_sequence(rows[0]))

### Many sequences: mid-frame grid

Better than any single clip for spotting a bad shard — cropping errors, stuck cameras, and
near-empty masks all pop out at a glance.

In [ ]:
sample = rows[:24]
fig, axes = plt.subplots(4, 6, figsize=(9, 9))
for ax, row in zip(axes.ravel(), sample):
    video = read_sequence(row)
    show_frame(ax, video[len(video) // 2], row["sequence_id"].split("/")[-1][:14])
for ax in axes.ravel()[len(sample):]:
    ax.axis("off")
fig.suptitle(f"{SHARD} — middle frame of first 24 sequences", fontsize=10)
fig.tight_layout()

### QC distributions

The inventory carries the statistics `validate_and_pack_sequence` computed, so you can plot the
shard's health and jump straight to outliers without decoding everything.

In [ ]:
foreground = np.array([float(row["foreground_fraction"]) for row in rows])
empty = np.array([float(row["empty_frame_fraction"]) for row in rows])
lengths = np.array([int(row["num_frames"]) for row in rows])

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, values, label in zip(
    axes, (foreground, empty, lengths), ("foreground_fraction", "empty_frame_fraction", "num_frames")
):
    ax.hist(values, bins=60)
    ax.set_xlabel(label)
    ax.set_ylabel("sequences")
fig.tight_layout()

print(f"foreground: median={np.median(foreground):.4f}  min={foreground.min():.5f}  max={foreground.max():.4f}")
print(f"frames:     median={int(np.median(lengths))}  min={lengths.min()}  max={lengths.max()}")

In [ ]:
# The thinnest silhouettes that still passed validation — usually the interesting failures.
for row in [rows[i] for i in np.argsort(foreground)[:3]]:
    contact_sheet(
        read_sequence(row),
        title=f"{row['sequence_id']}  fg={float(row['foreground_fraction']):.5f}",
    )

### Rejected sequences

Rows with `valid == "false"` have no offsets, so they cannot be read from the prepared shard —
only counted here, and viewed via the raw tar in section 3.

In [ ]:
from collections import Counter

all_rows = load_inventory(SHARD, valid_only=False)
rejected = [row for row in all_rows if row["valid"] != "true"]
print(f"{len(rejected)}/{len(all_rows)} sequences excluded")
for reason, count in Counter(row["exclusion_reason"].split(":")[0] for row in rejected).most_common():
    print(f"  {count:>6}  {reason}")

## 3. Raw `.tar.gz` shards

Only needed to see something the packer rejected.

**This unpickles data**, which executes arbitrary code — the same reason `pack_gaitlu_shard`
demands an explicit `trust_pickles=True`. Run it only against the trusted official release.

Stream sequentially with `r|gz`; random access into a gzip tar rescans from byte zero.

In [ ]:
from cody_jepa.data.gaitlu_prepare import _coerce_frame_array


def peek_raw(tar_gz, limit=4, name_filter=None):
    """Stream the first `limit` pickles (optionally matching a substring) out of a raw shard."""
    found = []
    with tarfile.open(tar_gz, mode="r|gz") as source:
        for member in source:
            if not member.isfile() or not member.name.endswith(".pkl"):
                continue
            if name_filter and name_filter not in member.name:
                continue
            payload = source.extractfile(member).read()
            value = pickle.load(io.BytesIO(payload))  # trusted release only
            found.append((member.name, _coerce_frame_array(value)))
            if len(found) >= limit:
                break
    return found

In [ ]:
for name, video in peek_raw(RAW / f"{SHARD}.tar.gz", limit=3):
    # Raw frames may be 0-255 rather than 0/1, so normalize before plotting.
    normalized = video.astype(np.float32) / max(float(video.max()), 1.0)
    print(name, video.shape, video.dtype, "max:", video.max())
    contact_sheet(normalized, title=name)

In [ ]:
# Inspect a specific rejected sequence by pulling it out of the raw shard.
if rejected:
    target = rejected[0]
    print(target["sequence_id"], "->", target["exclusion_reason"])
    for name, video in peek_raw(
        RAW / f"{target['source_shard'].removesuffix('.tar.gz')}.tar.gz",
        limit=1,
        name_filter=target["source_member"],
    ):
        normalized = video.astype(np.float32) / max(float(video.max()), 1.0)
        contact_sheet(normalized, title=f"REJECTED: {name}")

## 4. What the model actually sees

Sections 2 and 3 show sequences on disk. Neither shows a clip after temporal windowing, cropping,
and resizing. For that, go through the dataset itself — the same class training uses.

In [ ]:
from cody_jepa.data.gaitlu import GaitLUIndexedDataset

dataset = GaitLUIndexedDataset(
    find_manifest("val", preferred="common-holdout.csv"),
    data_root=PREPARED,
    split="val",
    clip_length=16,
    image_size=(112, 112),
)
print(len(dataset), "windows")

batch = dataset[0]
clip = batch["video"][:, 0].numpy()  # [T, 1, H, W] -> [T, H, W]
print(batch["sequence_id"], "window_start:", batch["window_start"], "clip:", clip.shape)
contact_sheet(clip, title=f"post-transform: {batch['sequence_id']}")

In [ ]:
# Augmentation sanity check: the same sequence under training-side random windows and crops.
train_view = GaitLUIndexedDataset(
    find_manifest("train", preferred="ladder-0-small.csv"),
    data_root=PREPARED,
    split="train",
    clip_length=16,
    image_size=(112, 112),
    random_windows=True,
    crop_scale=(0.7, 1.0),
    horizontal_flip_prob=0.5,
    deterministic_windows=1,
)

fig, axes = plt.subplots(3, 8, figsize=(12, 5))
for draw, axis_row in enumerate(axes):
    train_view.set_epoch(draw)
    sample = train_view[0]
    frames = sample["video"][:, 0].numpy()
    indices = np.linspace(0, len(frames) - 1, 8).astype(int)
    for ax, index in zip(axis_row, indices):
        show_frame(ax, frames[index], f"t={index}" if draw == 0 else None)
    axis_row[0].set_ylabel(f"epoch {draw}", fontsize=8)
fig.suptitle("same sequence, three augmented draws", fontsize=10)
fig.tight_layout()